
## 1. Delta Lake Fundamentals


## Q1. What is Delta Lake?


- Delta Lake is an `open-source storage framework` built on top of Parquet files. 
- It provides `ACID transactions, schema enforcement, schema evolution, time travel, MERGE operations` and` reliable data processing`.
- It helps us build `reliable data pipelines` on top of a data lake.


**Real-time example:**

In an E-Commerce project, we receive customer orders from multiple sources.

**Suppose an order has:**

- Order ID: 101
- Customer ID: 501
- Order Amount: ₹5,000
- Status: PLACED

- We store the incoming data in Bronze Delta tables, clean it in Silver Delta tables and aggregate it in Gold Delta tables.
- If a pipeline fails during an update, Delta Lake's transaction mechanism ensures that readers see a consistent committed table version rather than a partially committed transaction.



**Interview keywords:**

ACID, Parquet, transaction log, schema enforcement, time travel, MERGE.


## Q2. What is the difference between Data Lake, Delta Lake and Lakehouse?


| Concept | Explanation | Example |
|---|---|---|
| Data Lake | Stores raw structured and unstructured data | CSV, JSON, images, Parquet |
| Delta Lake | Adds reliability and transaction capabilities to data lake storage | Delta tables |
| Lakehouse | Combines data lake flexibility with data warehouse capabilities | Databricks Lakehouse |


**Real-time explanation:**

Imagine an E-Commerce company.
- **Data Lake:** Stores customer, order and payment files.
- **Delta Lake:** Provides reliable transactions and updates on those files.
- **Lakehouse:** Uses Bronze, Silver and Gold layers to support analytics, reporting and machine learning.


**Important:**

- **Delta Lake** is a storage framework. 
- **Lakehouse** is a broader architecture.


## Q3. What are the components of a Delta table?


**A Delta table mainly consists of:**

- Parquet data files.
- _delta_log transaction log.
- Checkpoint files.
- Optional deletion vector files and other transaction-related artifacts.


| Component | Purpose |
|---|---|
| Parquet files | Store actual table data |
| JSON transaction log | Records committed table changes |
| Checkpoint | Stores a consolidated snapshot of transaction state |
| Deletion vectors | Identify logically deleted or updated rows without immediately rewriting entire Parquet files |


**Important interview distinction:**

- A Delta table's transaction-managed storage consists of data files and transaction-log artifacts. 
- Unity Catalog adds governance and catalog metadata, such as table names, ownership and permissions.
- Unity Catalog metadata is not physically another component inside _delta_log.


## Q4. What is the difference between Parquet, ORC and Delta?


| Feature | Parquet | ORC | Delta |
|---|---|---|---|
| Type | Columnar file format | Columnar file format | Table/storage framework |
| Compression | Yes | Yes | Uses Parquet |
| ACID transactions | No | No | Yes |
| Time Travel | No | No | Yes |
| MERGE | Not natively | Not natively | Yes |
| Schema enforcement | No table-level transaction enforcement | No table-level transaction enforcement | Yes |
| Schema evolution | Reader-dependent | Reader-dependent | Supported |


**Interview answer:**

- Parquet and ORC are columnar storage formats. 
- Delta Lake uses Parquet for storing data and adds a transaction log to support ACID transactions, schema enforcement, time travel and MERGE operations.


## Q5. What are ACID properties in Delta Lake?


ACID stands for:

- A – Atomicity
- C – Consistency
- I – Isolation
- D – Durability

| Property | Meaning | Real-time Example |
|---|---|---|
| Atomicity | All changes succeed or fail together | Updating 1,000 orders |
| Consistency | Table remains valid after a transaction | Enforcing data types and constraints |
| Isolation | Concurrent operations don't expose partial changes | Two jobs updating orders |
| Durability | Committed changes remain available | Successfully committed payment records |


**Real-time scenario:**

- Suppose a payment pipeline updates 10,000 payment records. During processing, the job fails after preparing changes.
- Delta Lake ensures that an incomplete transaction is not exposed as a committed table version. 
- Readers continue to see the previous committed version.


## Q6. What is the Delta transaction log?


The Delta transaction log, stored in the _delta_log directory, tracks every committed change made to a Delta table.



**It records information such as:**

- Add file actions.
- Remove file actions.
- Metadata changes.
- Protocol changes.
- Transaction information.
Every successful transaction creates a new table version.


**Example:**

| Version | Operation | Description |
|---|---|---|
| 0 | CREATE | Created orders table |
| 1 | WRITE | Inserted 1,000 orders |
| 2 | UPDATE | Updated order status |
| 3 | DELETE | Deleted invalid orders |
| 4 | MERGE | Upserted new orders |



**Important:**

- The transaction log does not simply store the entire table data. 
- It stores the information required to reconstruct the active state of the table.


## Q7. What is a checkpoint in Delta Lake?


- A **checkpoint** is a consolidated snapshot of the transaction log state. 
- It helps Delta Lake avoid replaying every historical JSON transaction file when reconstructing the latest table state.

**Real-time example:**

- An orders table has thousands of transactions. 
- Checkpoints improve the efficiency of reconstructing the table snapshot.



**Interview follow-up:** 

**Does a checkpoint replace the JSON transaction log?**

- No. 
- JSON commit files continue to represent transactions. 
- Checkpoints provide a consolidated snapshot that speeds up log reconstruction.


## Q8. What are deletion vectors?


- Deletion vectors are a mechanism used by Delta Lake to identify rows that have been logically deleted or updated without immediately rewriting the complete underlying Parquet file.
- They can improve the performance of DELETE, UPDATE and MERGE operations.


**Real-time example:**

- A Parquet file contains 100,000 order records. We need to delete only 100 invalid orders.
- Instead of immediately rewriting the complete file, Delta Lake can use deletion vectors to mark the affected rows as deleted.


**Important:**

- Deletion vectors are not the same as physical deletion. 
- The underlying data can remain in storage until file-rewriting and retention cleanup processes remove it.


## Q9. What is schema enforcement?


- Schema enforcement ensures that incoming data follows the target Delta table's schema.
- If the target expects an integer column but incoming data contains incompatible values, the write can fail.


**Example:**


```python
orders_df.write\
    .format("delta")\
        .mode("append")\
            .saveAsTable("workspace.ecommerce_silver.orders")


**Suppose the target table contains:**

- order_id INT
- customer_id INT
- amount DOUBLE

- But incoming data has an incompatible amount type.
- Delta Lake validates the schema and can reject the write.



**Business example:**

Preventing invalid customer or payment records from entering trusted Silver tables


## Q10. What is schema evolution?


Schema evolution allows the schema of a Delta table to change when new data contains additional columns or compatible schema changes.

Example: Initially, the orders table contains:
order_id
customer_id
amount


Later, the business adds a new column:
delivery_status


We can enable schema evolution for the write:
orders_df.write \    .format("delta") \    .mode("append") \    .option("mergeSchema", "true") \    .saveAsTable("workspace.ecommerce_silver.orders")



Difference:
Schema Enforcement	Schema Evolution
Validates incoming schema	Allows supported schema changes
Prevents incompatible data writes	Supports new columns and compatible changes
Protects data quality	Supports changing business requirements

Interview tip: Don't say schema evolution accepts every schema mismatch. It supports specific compatible changes, and incompatible changes can still fail.

4. Time Travel


Q11. What is Time Travel in Delta Lake?
Interview answer:

Time Travel allows us to access previous versions of a Delta table using its transaction history.

Whenever a transaction commits, Delta Lake creates a new version.

We can access previous data using:
- Version number.
- Timestamp.
Real-time scenario:

Yesterday, the orders table had 10,000 records. Today, a developer accidentally deletes 2,000 records.

Using Time Travel, we can inspect the previous version and recover the data.
A. Check table history
DESCRIBE HISTORY workspace.ecommerce_silver.orders;


B. Read a previous version
old_orders_df = (    spark.read    .format("delta")    .option("versionAsOf", 2)    .table("workspace.ecommerce_silver.orders"))display(old_orders_df)



C. Read using timestamp
old_orders_df = (    spark.read    .format("delta")    .option("timestampAsOf", "2026-11-15 10:00:00")    .table("workspace.ecommerce_silver.orders"))display(old_orders_df)



D. Restore a previous version
RESTORE TABLE workspace.ecommerce_silver.orders
TO VERSION AS OF 2;


Important interview question: Does Time Travel mean we permanently store a complete copy of every table version?

No. Delta Lake uses transaction-log information and retained data files to reconstruct previous versions. It does not create a complete independent copy of the table for every transaction.

Important limitation: Time Travel works only while the required transaction-log information and data files are retained. VACUUM can remove files required to read older versions.

5. MERGE, UPSERT, UPDATE and DELETE
Q12. What is MERGE in Delta Lake?
Interview answer:

MERGE is used to perform INSERT, UPDATE and DELETE operations in a single transaction based on a matching condition between source and target tables.

It is commonly used in incremental data pipelines and CDC processing.

Real-time example:

Our E-Commerce application sends customer updates every day.

Existing customer:
customer_id	name	city
101	Arun	Chennai

Incoming source:
customer_id	name	city
101	Arun Kumar	Bangalore
102	Priya	Madurai

After MERGE:
customer_id	name	city
101	Arun Kumar	Bangalore
102	Priya	Madurai

Customer 101 is updated, and customer 102 is inserted.
MERGE syntax
MERGE INTO workspace.ecommerce_silver.customers AS target

USING workspace.ecommerce_bronze.customers AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;


UPDATE SET * and INSERT * require compatible source and target columns; use explicit column mappings when the schemas differ.
Q13. What is the difference between INSERT, UPDATE, DELETE and UPSERT?
Operation	Purpose	Example
INSERT	Adds new records	New customer
UPDATE	Modifies existing records	Customer changes city
DELETE	Removes matching records	Invalid customer
UPSERT	Updates existing and inserts new records	Daily customer feed

Example:
UPDATE workspace.ecommerce_silver.orders
SET order_status = 'DELIVERED'
WHERE order_id = 101;


DELETE FROM workspace.ecommerce_silver.orders
WHERE order_id = 999;


INSERT INTO workspace.ecommerce_silver.orders
VALUES (1001, 501, 5000.00, 'PLACED');


Q14. What happens if multiple source records match one target record in MERGE?
Interview answer:

If multiple source rows match the same target row for an update or delete, the MERGE can fail because the update is ambiguous.

We should deduplicate the source data before performing MERGE.

Example:
from pyspark.sql.window import Windowfrom pyspark.sql.functions import row_number, colwindow_spec = (    Window    .partitionBy("customer_id")    .orderBy(col("updated_at").desc()))deduplicated_df = (    source_df    .withColumn("rn", row_number().over(window_spec))    .filter(col("rn") == 1)    .drop("rn"))



Real-time example:

If customer 101 appears three times in the source, we select the latest updated record before merging.

This prevents ambiguous updates and ensures that the latest customer information is used.

6. SCD Type 1 and Type 2 – Very Important
Must Practise Hands-on
Q15. What is SCD?
SCD stands for Slowly Changing Dimension.

It is used in data warehouses to manage changes in dimension data over time.

Example: A customer changes their city from Chennai to Bangalore.

We need to decide whether to overwrite the old information or preserve historical information.

There are different SCD types, but Type 1 and Type 2 are especially important in Data Engineer interviews.
Q16. What is SCD Type 1?
Interview answer:

SCD Type 1 overwrites the existing record with the latest information. It does not maintain historical records.

Real-time example:

Before:
customer_id	name	city
101	Arun	Chennai

After customer moves:
customer_id	name	city
101	Arun	Bangalore

The old city is replaced.
Hands-on implementation: SCD Type 1
Step 1: Create target table
CREATE TABLE IF NOT EXISTS workspace.ecommerce_silver.customer_scd1
(
    customer_id INT,
    customer_name STRING,
    city STRING
)
USING DELTA;


Step 2: Insert initial records
INSERT INTO workspace.ecommerce_silver.customer_scd1
VALUES
(101, 'Arun', 'Chennai'),
(102, 'Priya', 'Madurai'),
(103, 'Kumar', 'Theni');


Step 3: Create source updates
CREATE OR REPLACE TEMP VIEW customer_updates AS

SELECT 101 AS customer_id,
       'Arun Kumar' AS customer_name,
       'Bangalore' AS city

UNION ALL

SELECT 104, 'Rahul', 'Coimbatore';


Step 4: Perform MERGE
MERGE INTO workspace.ecommerce_silver.customer_scd1 AS target

USING customer_updates AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET
        target.customer_name = source.customer_name,
        target.city = source.city

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        customer_name,
        city
    )
    VALUES (
        source.customer_id,
        source.customer_name,
        source.city
    );


Step 5: Verify
SELECT *
FROM workspace.ecommerce_silver.customer_scd1
ORDER BY customer_id;


Expected result:
customer_id	customer_name	city
101	Arun Kumar	Bangalore
102	Priya	Madurai
103	Kumar	Theni
104	Rahul	Coimbatore

Interview explanation:

I used MERGE to update existing customers and insert new customers. Since it is SCD Type 1, I overwrote the old customer information without maintaining historical records.

Q17. What is SCD Type 2?
Interview answer:

SCD Type 2 maintains complete historical records when dimension attributes change.

Instead of overwriting an existing record, we expire the old record and insert a new record.

We generally use:
- effective_start_date
- effective_end_date
- is_current
Real-time example:

Customer 101 changes city from Chennai to Bangalore.

Before:
customer_id	city	start_date	end_date	current
101	Chennai	2025-01-01	9999-12-31	TRUE

After:
customer_id	city	start_date	end_date	current
101	Chennai	2025-01-01	2026-11-15	FALSE
101	Bangalore	2026-11-15	9999-12-31	TRUE

Now we can answer both questions:
- What is the customer's current city? Bangalore.
- What was the customer's previous city? Chennai.
Hands-on implementation: SCD Type 2
Step 1: Create target table
CREATE TABLE IF NOT EXISTS workspace.ecommerce_silver.customer_scd2
(
    customer_id INT,
    customer_name STRING,
    city STRING,
    effective_start_date DATE,
    effective_end_date DATE,
    is_current BOOLEAN
)
USING DELTA;


Step 2: Insert initial data
INSERT INTO workspace.ecommerce_silver.customer_scd2
VALUES
(101, 'Arun', 'Chennai', '2025-01-01', '9999-12-31', TRUE),
(102, 'Priya', 'Madurai', '2025-01-01', '9999-12-31', TRUE);


Step 3: Prepare incoming source records
CREATE OR REPLACE TEMP VIEW customer_source AS

SELECT
    101 AS customer_id,
    'Arun Kumar' AS customer_name,
    'Bangalore' AS city

UNION ALL

SELECT
    103,
    'Rahul',
    'Coimbatore';


Step 4: Create staged records

The important concept here is that a changed existing customer must generate two logical actions:
1. Expire the existing record.
2. Insert the new record.
We use a staged MERGE source to achieve both.
CREATE OR REPLACE TEMP VIEW customer_scd2_staged AS

-- Expire existing records when customer details change
SELECT
    customer_id,
    customer_name,
    city,
    'UPDATE' AS merge_action
FROM (
    SELECT
        s.customer_id,
        s.customer_name,
        s.city
    FROM customer_source s
    INNER JOIN workspace.ecommerce_silver.customer_scd2 t
        ON s.customer_id = t.customer_id
    WHERE t.is_current = TRUE
      AND (
          NOT (s.customer_name <=> t.customer_name)
          OR NOT (s.city <=> t.city)
      )
) changed

UNION ALL

-- Insert new versions of changed and new customers
SELECT
    customer_id,
    customer_name,
    city,
    'INSERT' AS merge_action
FROM customer_source;


Step 5: Perform MERGE
MERGE INTO workspace.ecommerce_silver.customer_scd2 AS target

USING customer_scd2_staged AS source

ON target.customer_id = source.customer_id
AND target.is_current = TRUE
AND source.merge_action = 'UPDATE'

WHEN MATCHED THEN
UPDATE SET
    target.effective_end_date = date_sub(current_date(), 1),
    target.is_current = FALSE;


Step 6: Insert new versions
INSERT INTO workspace.ecommerce_silver.customer_scd2
(
    customer_id,
    customer_name,
    city,
    effective_start_date,
    effective_end_date,
    is_current
)

SELECT
    customer_id,
    customer_name,
    city,
    current_date(),
    DATE '9999-12-31',
    TRUE

FROM customer_scd2_staged
WHERE merge_action = 'INSERT';


Step 7: Verify
SELECT *
FROM workspace.ecommerce_silver.customer_scd2
ORDER BY customer_id, effective_start_date;


Expected logical result:
customer_id	customer_name	city	current
101	Arun	Chennai	FALSE
101	Arun Kumar	Bangalore	TRUE
102	Priya	Madurai	TRUE
103	Rahul	Coimbatore	TRUE

Important production considerations:
- The example uses current_date() for demonstration. Production pipelines should use the actual CDC event/effective timestamp.
- Run the expiration and insertion steps as a controlled, idempotent pipeline so a failure between steps cannot leave an incorrect current record.
- Handle late-arriving events and duplicate source records.
- Use a stable business key and validate that only one current record exists per customer.
Q18. Difference between SCD Type 1 and Type 2?
Feature	SCD Type 1	SCD Type 2
Historical data	Not maintained	Maintained
Existing record	Overwritten	Expired
New record	Only for new customers	For each new version
Storage	Less	More
Complexity	Simple	More complex
Use case	Correcting email spelling	Customer city history

Interview one-line answer:

SCD Type 1 overwrites existing data, whereas SCD Type 2 preserves historical changes by maintaining multiple versions of a record.

7. OPTIMIZE, ZORDER and Partitioning
Q19. What is OPTIMIZE in Delta Lake?
Interview answer:

OPTIMIZE is used to improve Delta table performance by reorganizing data files and reducing small-file problems.

It performs file compaction (bin-packing) by combining smaller files into larger, more efficient files.

Real-time example:

Suppose our E-Commerce orders table contains:
- 1,000 Parquet files.
- Each file is approximately 1 MB.
- Total data size is approximately 1 GB.
Having too many small files creates file-management and query overhead.

After OPTIMIZE, the data is reorganized into fewer, larger files.
SQL implementation
OPTIMIZE workspace.ecommerce_silver.orders;


Check table history:
DESCRIBE HISTORY workspace.ecommerce_silver.orders;


Important: OPTIMIZE does not change the logical table contents. It improves physical data layout. It also does not itself permanently delete the old files; VACUUM handles eligible obsolete files.
Q20. What is ZORDER in Delta Lake?
Interview answer:

ZORDER is a data-layout optimization technique that colocates related data values in the same set of files.

It improves data skipping, reducing the number of files that Spark needs to scan for selective queries.

Real-time example:

In our E-Commerce project, users frequently filter orders using:
- Customer ID.
- Order date.
We can use ZORDER to colocate data based on these columns.
SQL implementation
OPTIMIZE workspace.ecommerce_silver.orders
ZORDER BY (customer_id, order_date);


Now execute:
SELECT *
FROM workspace.ecommerce_silver.orders
WHERE customer_id = 501;


The optimized file layout can help Spark skip irrelevant files.

Interview tip: ZORDER does not create a traditional database index. It improves file-level data skipping through data colocation and statistics.
Q21. What is Partitioning in Delta Lake?
Interview answer:

Partitioning divides a large table into physical directory structures based on one or more partition columns.

It helps Spark eliminate irrelevant partitions when queries filter on those columns.

Real-time example:

Our orders table contains five years of data.

Most reports filter by order date.

We can partition by year and month.
PySpark implementation
orders_df.write \    .format("delta") \    .mode("overwrite") \    .partitionBy("order_year", "order_month") \    .saveAsTable("workspace.ecommerce_silver.orders_partitioned")



Possible physical layout:
orders_partitioned/
│
├── order_year=2025/
│   ├── order_month=01/
│   ├── order_month=02/
│   └── order_month=03/
│
└── order_year=2026/
    ├── order_month=01/
    ├── order_month=02/
    └── order_month=03/


Important interview question: Should we partition every Delta table?

No.

Excessive partitioning creates small files and metadata overhead. Partitioning is most useful when the table is sufficiently large and the selected columns have a sensible number of distinct values.

Current Databricks guidance generally favors Liquid Clustering for new tables rather than manually choosing partitioning and ZORDER strategies. 
Databricks on AWS+1

Q22. Difference between OPTIMIZE, ZORDER and Partitioning?
Feature	OPTIMIZE	ZORDER	Partitioning
Purpose	File compaction/layout	Data colocation	Physical data separation
Reduces small files	Yes	During optimization	Not necessarily
Improves data skipping	Through layout	Yes	Through partition pruning
Creates directories	No	No	Yes
Query performance	Improves	Can improve	Can improve
Used with	Delta tables	Non-liquid-clustered Delta tables	Partitioned tables

Remember:
- OPTIMIZE = File compaction.
- ZORDER = Data colocation.
- Partitioning = Physical directory separation.

8. Liquid Clustering – Important Modern Databricks Topic
High Priority – Modern Databricks
Q23. What is Liquid Clustering?
Interview answer:

Liquid Clustering is a modern data-layout optimization technique in Delta Lake that replaces the need for traditional partitioning and ZORDER in many workloads.

It organizes data based on selected clustering columns and allows us to change those columns as query patterns evolve.

Unlike traditional partitioning, it does not create fixed directory partitions.

Real-time example:

Our E-Commerce orders table contains 20 million records.

Business analysts frequently query:
SELECT *
FROM workspace.ecommerce_silver.orders
WHERE customer_id = 501;


And:
SELECT *
FROM workspace.ecommerce_silver.orders
WHERE order_date = '2026-11-15';


We can cluster the table using:
- customer_id
- order_date
Create a Liquid Clustered table
CREATE TABLE workspace.ecommerce_silver.orders_liquid
USING DELTA
CLUSTER BY (customer_id, order_date)
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


Enable Liquid Clustering on an existing table
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (customer_id, order_date);


Then reorganize the existing data:
OPTIMIZE workspace.ecommerce_silver.orders FULL;


OPTIMIZE FULL is supported in Databricks Runtime 16.0 and above.

Important: Changing clustering columns does not automatically recluster all previously written files. Run OPTIMIZE to apply the new layout. 
Databricks on AWS+1

Q24. What are the advantages of Liquid Clustering?
1. No fixed partition directories.
2. Supports high-cardinality columns.
3. Clustering keys can be changed.
4. Improves data skipping.
5. Reduces manual data-layout management.
6. Works well for evolving query patterns.
Real-time example:

Initially, business analysts query using customer_id.

Later, they start querying using product_id.

With traditional partitioning, changing the partition strategy can require expensive data rewrites.

With Liquid Clustering, we can change the clustering keys and run OPTIMIZE.
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (product_id, customer_id);

OPTIMIZE workspace.ecommerce_silver.orders FULL;


Q25. Difference between ZORDER and Liquid Clustering?
Feature	ZORDER	Liquid Clustering
Data organization	Multi-column colocation	Multi-dimensional clustering
Key changes	Requires a new optimization strategy	Keys can be changed using ALTER
Traditional partitioning	Can be combined with partitions	Designed to replace traditional partitioning
Small-file compaction	Through OPTIMIZE	Through OPTIMIZE
New table recommendation	Legacy choice for many workloads	Preferred for new Databricks tables
ZORDER BY syntax	Supported	Not supported together

Important correction for interviews:

Liquid Clustering does not eliminate OPTIMIZE.

It replaces the need for manually combining partitioning and ZORDER as the primary data-layout strategy. OPTIMIZE is still used to organize and recluster data.
Q26. How do you convert an existing ZORDER table to Liquid Clustering?
Suppose our existing table is:
OPTIMIZE workspace.ecommerce_silver.orders
ZORDER BY (customer_id, order_date);


We want to migrate to Liquid Clustering.

Step 1: Enable Liquid Clustering
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (customer_id, order_date);


Step 2: Recluster existing files
OPTIMIZE workspace.ecommerce_silver.orders FULL;


Step 3: Verify table configuration
DESCRIBE DETAIL workspace.ecommerce_silver.orders;


Inspect the clustering information and table metadata.

Important migration distinction:
- If the existing table is ZORDER-only and unpartitioned, use the ALTER and OPTIMIZE approach above.
- If it is also partitioned, modern Databricks Runtime 18.1+ supports REPLACE PARTITIONED BY WITH CLUSTER BY for conversion.
Example:
ALTER TABLE workspace.ecommerce_silver.orders
REPLACE PARTITIONED BY WITH CLUSTER BY
(customer_id, order_date);


Then:
OPTIMIZE workspace.ecommerce_silver.orders FULL;


This is version-dependent; do not claim that the partitioned-table conversion command works on every Databricks Runtime. 
Databricks on AWS+1

Q27. What is the difference between CLUSTER BY and partitionBy?
Feature	partitionBy	CLUSTER BY
Technology	Traditional partitioning	Liquid Clustering
Directory structure	Creates partition directories	No fixed partition directories
Changing columns	Can require expensive rewrites	Easier to change
High-cardinality fields	Usually unsuitable	Supported
Data layout	Fixed partition boundaries	Flexible clustering
Recommendation	Specific suitable workloads	Preferred for new Delta tables

Example:
# Traditional partitioningorders_df.write \    .format("delta") \    .partitionBy("order_year") \    .saveAsTable("workspace.ecommerce_silver.orders_partitioned")



Liquid Clustering:
CREATE TABLE workspace.ecommerce_silver.orders_clustered
CLUSTER BY (customer_id, order_date)
AS
SELECT *
FROM workspace.ecommerce_silver.orders;



9. VACUUM – Storage Cleanup
Q28. What is VACUUM in Delta Lake?
Interview answer:

VACUUM is used to remove obsolete data files that are no longer required by the retained Delta table versions.

It helps reduce storage costs.

Real-time example:

Our orders table originally contains 1 million records.

We update 10,000 records.

Delta Lake may create new data files and mark previous files as removed from the latest table snapshot.

The old files can remain in cloud storage for Time Travel.

After the retention period, VACUUM can permanently remove eligible obsolete files.
SQL implementation
VACUUM workspace.ecommerce_silver.orders;


Default retention:
168 hours = 7 days
Preview files before deleting:
VACUUM workspace.ecommerce_silver.orders DRY RUN;


Vacuum with a specified retention period:
VACUUM workspace.ecommerce_silver.orders
RETAIN 168 HOURS;


Important interview questions:

1. Does VACUUM delete the latest active data files?

No. It removes files that are no longer required by the retained table state and are eligible under the retention policy.

2. Does VACUUM delete _delta_log JSON files?

No. VACUUM is not responsible for normal transaction-log retention cleanup. Log cleanup is handled separately.

3. What happens if we run VACUUM with zero hours?

It can remove files needed by concurrent or older readers and destroy the ability to time travel to older versions. Databricks has a safety check against unsafe retention periods.

4. What is the default retention of the transaction log?

The default delta.logRetentionDuration is 30 days, whereas the default deleted-data-file retention is 7 days. These are separate settings. 
Databricks on AWS+1

Time Travel vs VACUUM
Time Travel	VACUUM
Reads previous versions	Removes eligible obsolete files
Supports historical analysis	Reduces storage usage
Uses retained files and log information	Physically deletes eligible files
Can recover historical data	Can make older versions unreadable

Production advice: Never reduce the 7-day retention period just to save storage without understanding long-running jobs, concurrent readers, streaming workloads and recovery requirements.

10. CTAS, DEEP CLONE and SHALLOW CLONE
Q29. What is CTAS in Databricks?
CTAS stands for Create Table As Select.

It creates a new table using the results of a SELECT query.

Example:
CREATE TABLE workspace.ecommerce_silver.orders_ctas
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


Real-time example:

We have a production orders table and need a separate table containing only completed orders.
CREATE TABLE workspace.ecommerce_silver.completed_orders
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders
WHERE order_status = 'COMPLETED';


CTAS creates a new dataset from the query results.

It does not automatically preserve all source table properties, constraints, permissions or historical versions.
Q30. What is DEEP CLONE?
Interview answer:

DEEP CLONE creates an independent copy of a Delta table, including its data files and table metadata.

The cloned table can be modified independently without affecting the source table.

Real-time example:

Our production orders table contains 10 million records.

Before testing a major transformation, we create a development copy.
CREATE TABLE workspace.ecommerce_silver.orders_deep_clone
DEEP CLONE workspace.ecommerce_silver.orders;


Advantages:
- Independent data.
- Independent table history.
- Suitable for testing and migration.
- Does not depend on the source table's data files after cloning.
Disadvantage:
- Requires additional storage for copied data.
- More expensive than shallow cloning.
Q31. What is SHALLOW CLONE?
Interview answer:

SHALLOW CLONE creates a new Delta table that shares or references the source table's existing data files instead of copying all the data files.

It creates independent table metadata and history.

Example:
CREATE TABLE workspace.ecommerce_silver.orders_shallow_clone
SHALLOW CLONE workspace.ecommerce_silver.orders;


Advantages:
- Faster cloning.
- Lower initial storage requirements.
- Useful for development and testing.
Disadvantages:
- Depends on source data files.
- VACUUM or source-file cleanup can affect the clone's ability to read shared files, depending on the clone type and catalog implementation.
Real-time example:

A developer wants to test a query against production orders without creating a complete physical copy.

Shallow clone can be useful for this situation.
Q32. Difference between CTAS, DEEP CLONE and SHALLOW CLONE?
Feature	CTAS	DEEP CLONE	SHALLOW CLONE
Data copied	Query result materialized	Yes	No full copy
Source metadata preserved	Not fully	Yes	Yes
Independent history	New history	Yes	Yes
Initial storage cost	Depends on result size	High	Low
Source dependency	No	No	Yes, for shared data files
Main use case	Create transformed table	Independent backup/testing copy	Fast development copy

Very important interview point:

A clone has independent table history. If the source table is at version 10, the clone's history does not automatically become the same history with version 10. 
Databricks on AWS+1


Q33. Hands-on: Create one table using CTAS, DEEP CLONE and SHALLOW CLONE and compare storage and metadata.
First create a source table:
CREATE TABLE workspace.ecommerce_silver.orders_source
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


1. CTAS
CREATE TABLE workspace.ecommerce_silver.orders_ctas
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders_source;


2. DEEP CLONE
CREATE TABLE workspace.ecommerce_silver.orders_deep
DEEP CLONE workspace.ecommerce_silver.orders_source;


3. SHALLOW CLONE
CREATE TABLE workspace.ecommerce_silver.orders_shallow
SHALLOW CLONE workspace.ecommerce_silver.orders_source;


4. Compare row counts
SELECT 'SOURCE' AS table_name, COUNT(*) AS total_rows
FROM workspace.ecommerce_silver.orders_source

UNION ALL

SELECT 'CTAS', COUNT(*)
FROM workspace.ecommerce_silver.orders_ctas

UNION ALL

SELECT 'DEEP CLONE', COUNT(*)
FROM workspace.ecommerce_silver.orders_deep

UNION ALL

SELECT 'SHALLOW CLONE', COUNT(*)
FROM workspace.ecommerce_silver.orders_shallow;


5. Compare table metadata
DESCRIBE DETAIL workspace.ecommerce_silver.orders_source;


DESCRIBE DETAIL workspace.ecommerce_silver.orders_ctas;


DESCRIBE DETAIL workspace.ecommerce_silver.orders_deep;


DESCRIBE DETAIL workspace.ecommerce_silver.orders_shallow;


Compare:
- Format.
- Location.
- Number of files.
- Size in bytes.
- Partition columns.
- Clustering information, where applicable.
Expected comparison
Metric	CTAS	Deep Clone	Shallow Clone
Data files	New	Copied	Shared/referenced
Delta metadata	New	Copied/preserved	New clone metadata
Storage requirement	New data storage	Additional data storage	Lower initial requirement
Independent data modifications	Yes	Yes	Yes, as a separate table
Depends on source files	No	No	Yes

Note: The actual storage usage depends on source size, clone behavior, storage format, incremental changes and Unity Catalog implementation. Do not expect identical byte counts in every environment.

11. Change Data Feed (CDF)
Very Important – Incremental ETL
Q34. What is Change Data Feed in Delta Lake?
Interview answer:

Change Data Feed (CDF) captures row-level changes made to a Delta table, including INSERT, UPDATE and DELETE operations.

It allows downstream pipelines to process only changed records instead of reading the entire source table repeatedly.

Real-time example:

Our E-Commerce application has a customers table.

Initially:
customer_id	name	city
101	Arun	Chennai
102	Priya	Madurai

Next day:
- Customer 101 changes city to Bangalore.
- Customer 102 is deleted.
- Customer 103 is inserted.
Instead of processing the entire customers table, CDF helps us identify and process these changes.
Q35. How do you enable Change Data Feed?
Step 1: Enable CDF
ALTER TABLE workspace.ecommerce_silver.customers
SET TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);


Important: This enables the legacy, write-time-materialized CDF for subsequent changes. It does not retroactively capture changes made before enabling it.

Step 2: Insert a record
INSERT INTO workspace.ecommerce_silver.customers
VALUES (103, 'Rahul', 'Coimbatore');


Step 3: Update a record
UPDATE workspace.ecommerce_silver.customers
SET city = 'Bangalore'
WHERE customer_id = 101;


Step 4: Delete a record
DELETE FROM workspace.ecommerce_silver.customers
WHERE customer_id = 102;


Step 5: Read CDF changes
cdf_df = (    spark.read    .format("delta")    .option("readChangeFeed", "true")    .option("startingVersion", 1)    .table("workspace.ecommerce_silver.customers"))display(cdf_df)



CDF returns additional metadata columns:
Column	Purpose
_change_type	Type of change
_commit_version	Version in which change occurred
_commit_timestamp	Commit timestamp

Possible _change_type values:
- insert
- update_preimage
- update_postimage
- delete
Important: An UPDATE can produce two records: the old row (update_preimage) and the new row (update_postimage).
Q36. How do you use CDF in a real-time incremental pipeline?
Suppose the customers table is our source and the customer_audit table is our downstream table.
cdf_df = (    spark.read    .format("delta")    .option("readChangeFeed", "true")    .option("startingVersion", 5)    .table("workspace.ecommerce_silver.customers"))



Filter inserted and updated records:
from pyspark.sql.functions import colchanged_records = cdf_df.filter(    col("_change_type").isin(        "insert",        "update_postimage"    ))



Read deleted records:
deleted_records = cdf_df.filter(    col("_change_type") == "delete")



Write the changes into an audit table:
(    cdf_df.write    .format("delta")    .mode("append")    .saveAsTable("workspace.ecommerce_silver.customer_audit"))



Production consideration: The example demonstrates batch reading. A production incremental pipeline should track its last successfully processed version, handle retries idempotently and process deletes and update preimages/postimages according to business requirements.

CDF is not a permanent audit archive by itself. Its data is subject to retention, so archive changes into another durable table if permanent history is required. 
Databricks on AWS+1

Q37. Difference between CDF and Time Travel?
Feature	CDF	Time Travel
Purpose	Captures row-level changes	Reads previous table snapshots
Output	Changed rows	Complete table snapshot
Insert detection	Yes	Not directly
Update detection	Yes	Not directly
Delete detection	Yes	Not directly
Main use	Incremental ETL, CDC, auditing	Historical queries, recovery

Simple interview answer:

Time Travel helps us read an older version of a table, whereas CDF helps us identify which records changed between table versions.

12. Frequently Asked Advanced Interview Questions
These are additional questions you should prepare for a 2-year Data Engineer interview.
Q38. What happens when an UPDATE operation runs in Delta Lake?
Delta Lake creates a new committed table version.

Depending on the enabled table features and execution strategy, it can rewrite affected data files or use deletion vectors to mark affected rows and write replacement data.

The transaction log records the changes.

The old data files can remain available for historical reads until retention cleanup.
Q39. Can we use UPDATE and DELETE in Parquet?
Parquet is a file format and does not provide native transactional UPDATE and DELETE operations.

We generally need to rewrite files or use another processing framework to implement such changes.

Delta Lake provides transactional UPDATE and DELETE operations.
Q40. What is idempotency in Delta Lake?
Idempotency means executing the same operation multiple times should produce the same final result without creating unintended duplicate data or incorrect updates.

Real-time example:

An orders ingestion job fails after processing 5,000 records.

When the job restarts, we don't want those 5,000 records inserted again.

We use stable business keys, deduplication, checkpoints and MERGE logic to avoid duplicates.
Q41. Can multiple users access a Delta table simultaneously?
Yes.

Delta Lake supports concurrent reads and writes using its transaction protocol and isolation mechanisms.

Readers see consistent snapshots, and concurrent writes are validated for conflicts.

If two jobs modify conflicting data, one transaction may need to retry.
Q42. What is data skipping?
Data skipping allows Spark to skip irrelevant data files using file-level statistics such as minimum and maximum column values.

Example:
SELECT *
FROM workspace.ecommerce_silver.orders
WHERE customer_id = 501;


If file statistics indicate that a file cannot contain customer ID 501, Spark can skip scanning that file.

ZORDER and Liquid Clustering can improve data skipping by organizing related records more effectively.
Q43. What is the difference between OPTIMIZE and VACUUM?
OPTIMIZE	VACUUM
Reorganizes data files	Removes eligible obsolete files
Improves query performance	Reduces storage usage
Performs compaction/reclustering	Performs physical cleanup
Creates a new table version	Does not represent a normal data update
Does not permanently clean all old files	Permanently deletes eligible files

Q44. What is the difference between Delta table history and Time Travel?
DESCRIBE HISTORY displays the history of table operations.

Time Travel reads the actual table data at a particular historical version.

Example:
DESCRIBE HISTORY workspace.ecommerce_silver.orders;


SELECT *
FROM workspace.ecommerce_silver.orders
VERSION AS OF 5;


Q45. Can we recover deleted data after VACUUM?
If VACUUM has permanently deleted the Parquet files needed for the historical version, Delta Time Travel cannot reconstruct those records from the transaction log alone.

Recovery requires another available backup, clone, archive or replicated copy.

This is why retention policies must be designed carefully.

13. Real-Time E-Commerce Lakehouse Implementation
Let's connect all these concepts to one project.








quicklearnpro.com


Business scenario:

An E-Commerce company receives customer, product, order, payment and shipment data.

We implement the following:
Layer	Implementation	Delta concept
Bronze	Store incoming source records	Delta WRITE
Silver	Clean and deduplicate records	Schema enforcement
Silver	Update customer details	MERGE
Silver	Maintain customer history	SCD Type 2
Silver	Capture source changes	CDF
Silver	Improve data layout	Liquid Clustering
Gold	Build sales aggregations	OPTIMIZE
Gold	Historical reporting	Time Travel
Maintenance	Remove old files	VACUUM

Complete workflow
#chatgpt-mermaid-_r_1ef_{font-family:-apple-system-body,ui-sans-serif,-apple-system,system-ui,"Segoe UI",Helvetica,"Apple Color Emoji",Arial,sans-serif,"Segoe UI Emoji","Segoe UI Symbol";font-size:16px;fill:rgb(237, 237, 237);}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#chatgpt-mermaid-_r_1ef_ .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#chatgpt-mermaid-_r_1ef_ .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#chatgpt-mermaid-_r_1ef_ .error-icon{fill:rgb(48, 48, 48);}#chatgpt-mermaid-_r_1ef_ .error-text{fill:rgb(237, 237, 237);stroke:rgb(237, 237, 237);}#chatgpt-mermaid-_r_1ef_ .edge-thickness-normal{stroke-width:1px;}#chatgpt-mermaid-_r_1ef_ .edge-thickness-thick{stroke-width:3.5px;}#chatgpt-mermaid-_r_1ef_ .edge-pattern-solid{stroke-dasharray:0;}#chatgpt-mermaid-_r_1ef_ .edge-thickness-invisible{stroke-width:0;fill:none;}#chatgpt-mermaid-_r_1ef_ .edge-pattern-dashed{stroke-dasharray:3;}#chatgpt-mermaid-_r_1ef_ .edge-pattern-dotted{stroke-dasharray:2;}#chatgpt-mermaid-_r_1ef_ .marker{fill:rgb(175, 175, 175);stroke:rgb(175, 175, 175);}#chatgpt-mermaid-_r_1ef_ .marker.cross{stroke:rgb(175, 175, 175);}#chatgpt-mermaid-_r_1ef_ svg{font-family:-apple-system-body,ui-sans-serif,-apple-system,system-ui,"Segoe UI",Helvetica,"Apple Color Emoji",Arial,sans-serif,"Segoe UI Emoji","Segoe UI Symbol";font-size:16px;}#chatgpt-mermaid-_r_1ef_ p{margin:0;}#chatgpt-mermaid-_r_1ef_ .label{font-family:-apple-system-body,ui-sans-serif,-apple-system,system-ui,"Segoe UI",Helvetica,"Apple Color Emoji",Arial,sans-serif,"Segoe UI Emoji","Segoe UI Symbol";color:rgb(237, 237, 237);}#chatgpt-mermaid-_r_1ef_ .cluster-label text{fill:rgb(237, 237, 237);}#chatgpt-mermaid-_r_1ef_ .cluster-label span{color:rgb(237, 237, 237);}#chatgpt-mermaid-_r_1ef_ .cluster-label span p{background-color:transparent;}#chatgpt-mermaid-_r_1ef_ .label text,#chatgpt-mermaid-_r_1ef_ span{fill:rgb(237, 237, 237);color:rgb(237, 237, 237);}#chatgpt-mermaid-_r_1ef_ .node rect,#chatgpt-mermaid-_r_1ef_ .node circle,#chatgpt-mermaid-_r_1ef_ .node ellipse,#chatgpt-mermaid-_r_1ef_ .node polygon,#chatgpt-mermaid-_r_1ef_ .node path{fill:rgb(9, 23, 44);stroke:rgb(31, 78, 148);stroke-width:1px;}#chatgpt-mermaid-_r_1ef_ .rough-node .label text,#chatgpt-mermaid-_r_1ef_ .node .label text,#chatgpt-mermaid-_r_1ef_ .image-shape .label,#chatgpt-mermaid-_r_1ef_ .icon-shape .label{text-anchor:middle;}#chatgpt-mermaid-_r_1ef_ .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#chatgpt-mermaid-_r_1ef_ .rough-node .label,#chatgpt-mermaid-_r_1ef_ .node .label,#chatgpt-mermaid-_r_1ef_ .image-shape .label,#chatgpt-mermaid-_r_1ef_ .icon-shape .label{text-align:center;}#chatgpt-mermaid-_r_1ef_ .node.clickable{cursor:pointer;}#chatgpt-mermaid-_r_1ef_ .root .anchor path{fill:rgb(175, 175, 175)!important;stroke-width:0;stroke:rgb(175, 175, 175);}#chatgpt-mermaid-_r_1ef_ .arrowheadPath{fill:rgb(175, 175, 175);}#chatgpt-mermaid-_r_1ef_ .edgePath .path{stroke:rgb(175, 175, 175);stroke-width:1px;}#chatgpt-mermaid-_r_1ef_ .flowchart-link{stroke:rgb(175, 175, 175);fill:none;}#chatgpt-mermaid-_r_1ef_ .edgeLabel{background-color:rgb(0, 0, 0);text-align:center;}#chatgpt-mermaid-_r_1ef_ .edgeLabel p{background-color:rgb(0, 0, 0);}#chatgpt-mermaid-_r_1ef_ .edgeLabel rect{opacity:0.5;background-color:rgb(0, 0, 0);fill:rgb(0, 0, 0);}#chatgpt-mermaid-_r_1ef_ .labelBkg{background-color:rgba(0, 0, 0, 0.5);}#chatgpt-mermaid-_r_1ef_ .cluster rect{fill:rgb(48, 48, 48);stroke:rgba(255, 255, 255, 0.15);stroke-width:1px;}#chatgpt-mermaid-_r_1ef_ .cluster text{fill:rgb(237, 237, 237);}#chatgpt-mermaid-_r_1ef_ .cluster span{color:rgb(237, 237, 237);}#chatgpt-mermaid-_r_1ef_ div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:-apple-system-body,ui-sans-serif,-apple-system,system-ui,"Segoe UI",Helvetica,"Apple Color Emoji",Arial,sans-serif,"Segoe UI Emoji","Segoe UI Symbol";font-size:12px;background:rgb(48, 48, 48);border:1px solid rgba(255, 255, 255, 0.15);border-radius:2px;pointer-events:none;z-index:100;}#chatgpt-mermaid-_r_1ef_ .flowchartTitleText{text-anchor:middle;font-size:18px;fill:rgb(237, 237, 237);}#chatgpt-mermaid-_r_1ef_ rect.text{fill:none;stroke-width:0;}#chatgpt-mermaid-_r_1ef_ .icon-shape,#chatgpt-mermaid-_r_1ef_ .image-shape{background-color:rgb(0, 0, 0);text-align:center;}#chatgpt-mermaid-_r_1ef_ .icon-shape p,#chatgpt-mermaid-_r_1ef_ .image-shape p{background-color:rgb(0, 0, 0);padding:2px;}#chatgpt-mermaid-_r_1ef_ .icon-shape .label rect,#chatgpt-mermaid-_r_1ef_ .image-shape .label rect{opacity:0.5;background-color:rgb(0, 0, 0);fill:rgb(0, 0, 0);}#chatgpt-mermaid-_r_1ef_ .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#chatgpt-mermaid-_r_1ef_ .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#chatgpt-mermaid-_r_1ef_ .node .neo-node{stroke:rgb(31, 78, 148);}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].node rect,#chatgpt-mermaid-_r_1ef_ [data-look="neo"].cluster rect,#chatgpt-mermaid-_r_1ef_ [data-look="neo"].node polygon{stroke:url(#chatgpt-mermaid-_r_1ef_-gradient);filter:drop-shadow( 1px 2px 2px rgba(185,185,185,1));}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].swimlane.cluster rect{filter:none;}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].node path{stroke:url(#chatgpt-mermaid-_r_1ef_-gradient);stroke-width:1px;}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].node .outer-path{filter:drop-shadow( 1px 2px 2px rgba(185,185,185,1));}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].node .neo-line path{stroke:rgb(31, 78, 148);filter:none;}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].node circle{stroke:url(#chatgpt-mermaid-_r_1ef_-gradient);filter:drop-shadow( 1px 2px 2px rgba(185,185,185,1));}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].node circle .state-start{fill:#000000;}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].icon-shape .icon{fill:url(#chatgpt-mermaid-_r_1ef_-gradient);filter:drop-shadow( 1px 2px 2px rgba(185,185,185,1));}#chatgpt-mermaid-_r_1ef_ [data-look="neo"].icon-shape .icon-neo path{stroke:url(#chatgpt-mermaid-_r_1ef_-gradient);filter:drop-shadow( 1px 2px 2px rgba(185,185,185,1));}#chatgpt-mermaid-_r_1ef_ .node text{font-size:14px;font-weight:600;letter-spacing:normal;fill:rgb(153, 206, 255);}#chatgpt-mermaid-_r_1ef_ .edgeLabels text{font-size:13px;font-weight:600;letter-spacing:-0.08px;fill:rgb(153, 206, 255);}#chatgpt-mermaid-_r_1ef_ .node tspan[font-weight="normal"],#chatgpt-mermaid-_r_1ef_ .edgeLabels tspan[font-weight="normal"]{font-weight:600;}#chatgpt-mermaid-_r_1ef_ .edgeLabel .label rect{opacity:1;rx:13px;ry:13px;fill:rgb(0, 14, 26);stroke:rgb(26, 62, 95);stroke-width:1px;}#chatgpt-mermaid-_r_1ef_ .node rect,#chatgpt-mermaid-_r_1ef_ .node circle,#chatgpt-mermaid-_r_1ef_ .node ellipse,#chatgpt-mermaid-_r_1ef_ .node polygon,#chatgpt-mermaid-_r_1ef_ .node path{fill:rgb(0, 40, 77);stroke:rgba(255, 255, 255, 0.1);stroke-width:1px;}#chatgpt-mermaid-_r_1ef_ .node rect{rx:16px;ry:16px;}#chatgpt-mermaid-_r_1ef_ .mermaid-decision .label-container{fill:rgb(0, 14, 26);stroke:rgb(26, 62, 95);stroke-dasharray:2,2;}#chatgpt-mermaid-_r_1ef_ .edgePaths .flowchart-link{stroke:rgb(175, 175, 175);stroke-width:1px;stroke-linecap:round;stroke-linejoin:round;}#chatgpt-mermaid-_r_1ef_ .marker{fill:rgb(175, 175, 175);stroke:rgb(175, 175, 175);}#chatgpt-mermaid-_r_1ef_ :root{--mermaid-font-family:-apple-system-body,ui-sans-serif,-apple-system,system-ui,"Segoe UI",Helvetica,"Apple Color Emoji",Arial,sans-serif,"Segoe UI Emoji","Segoe UI Symbol";}E-Commerce Source DataBronze Delta TablesSchema EnforcementSilver Data CleaningDeduplicationMERGE / UPSERTSCD Type 1 / Type 2CDF Change CaptureGold Delta TablesOPTIMIZE / Liquid ClusteringSQL DashboardsBusiness AnalyticsTime Travel / Transaction LogVACUUM / Retention





How to explain this project in an interview
Interviewer: Explain where you used Delta Lake in your E-Commerce project.

Your answer:

"I implemented Delta Lake as the storage layer for my E-Commerce Lakehouse project using Databricks and PySpark.

I used Delta tables across Bronze, Silver and Gold layers.

In Bronze, I stored incoming source data in Delta format.

In Silver, I implemented schema enforcement, data cleansing, deduplication and MERGE operations to handle incremental records.

For customer dimension management, I implemented SCD Type 1 and Type 2. Type 1 overwrote existing customer information, whereas Type 2 maintained historical customer records using effective dates and current-record flags.

I also studied Change Data Feed for incremental processing, Time Travel for historical recovery and table auditing, and OPTIMIZE and Liquid Clustering for query performance.

Finally, I used VACUUM concepts for storage maintenance and created Gold tables for business dashboards."

One important honesty point: Only claim that you implemented a feature hands-on if you actually ran and verified it. Distinguish between implemented, tested and studied features. Interviewers can quickly expose inflated project experience through follow-up questions.

14. Last-Minute Interview Revision Sheet
Memorize these distinctions before the interview.
Interview keyword	One-line answer
Delta Lake	Reliable storage framework built on Parquet
ACID	Atomicity, Consistency, Isolation, Durability
_delta_log	Tracks committed table transactions
Checkpoint	Consolidated transaction-log snapshot
Deletion vectors	Mark affected rows without immediately rewriting entire files
Schema enforcement	Validates incoming data schema
Schema evolution	Allows supported schema changes
MERGE	Performs conditional INSERT, UPDATE and DELETE
SCD Type 1	Overwrites old records
SCD Type 2	Maintains historical records
Time Travel	Reads previous table versions
OPTIMIZE	Compacts or reorganizes data files
ZORDER	Colocates related data
Partitioning	Separates data into physical partitions
Liquid Clustering	Flexible data clustering without fixed partition directories
VACUUM	Removes eligible obsolete files
CTAS	Creates a table from a SELECT query
DEEP CLONE	Copies data and table metadata
SHALLOW CLONE	Creates a clone referencing shared data files
CDF	Captures row-level changes
Data skipping	Skips irrelevant data files

Important SQL commands – Quick revision
-- 1. Table history
DESCRIBE HISTORY workspace.ecommerce_silver.orders;

-- 2. Table metadata
DESCRIBE DETAIL workspace.ecommerce_silver.orders;

-- 3. Time Travel
SELECT *
FROM workspace.ecommerce_silver.orders
VERSION AS OF 5;

-- 4. Restore
RESTORE TABLE workspace.ecommerce_silver.orders
TO VERSION AS OF 5;

-- 5. MERGE
MERGE INTO target
USING source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


Delta Lake – Complete Hands-on Practice (12 Tasks)
For a Data Engineer targeting 2 years of experience, don't stop at writing the commands. You should be able to explain what happens internally, why you use each operation, and what can go wrong in production.

The examples below use Databricks SQL and your E-Commerce Lakehouse catalog, workspace. Run them in order. They use a separate practice schema so you don't accidentally modify your project tables.
0. Create a practice schema
CREATE SCHEMA IF NOT EXISTS workspace.delta_practice;

USE CATALOG workspace;

USE SCHEMA delta_practice;


Create a sample customer table:
CREATE OR REPLACE TABLE customers (
    customer_id INT,
    customer_name STRING,
    city STRING,
    email STRING,
    updated_at TIMESTAMP
)
USING DELTA;


Insert sample records:
INSERT INTO customers VALUES
(1, 'Arun', 'Chennai', 'arun@gmail.com', current_timestamp()),
(2, 'Bala', 'Madurai', 'bala@gmail.com', current_timestamp()),
(3, 'Charan', 'Coimbatore', 'charan@gmail.com', current_timestamp()),
(4, 'David', 'Theni', 'david@gmail.com', current_timestamp());


Check the data:
SELECT * FROM customers;


Expected: four customer records.

1. Create a Delta table and inspect _delta_log
Objective: Understand how Delta Lake tracks table changes.

A Delta table stores its data in Parquet files and maintains a transaction log in the _delta_log directory. The transaction log records committed table operations and the metadata needed to reconstruct a consistent table version.
DESCRIBE DETAIL customers;


Check the table history:
DESCRIBE HISTORY customers;


Read the underlying files:
DESCRIBE DETAIL customers;


The result includes the table location. In a supported environment, inspect the storage directory using that location:
display(dbutils.fs.ls("YOUR_TABLE_LOCATION"))



To inspect the transaction log:
display(dbutils.fs.ls("YOUR_TABLE_LOCATION/_delta_log"))



You may see JSON commit files and checkpoint files. Exact filenames and file contents depend on the table's history and runtime.

What happens internally?
1. Data is written to Parquet files.
2. A transaction is prepared.
3. A new commit is recorded in _delta_log.
4. Readers use the transaction log to identify the valid snapshot of the table.
Interview answer: Delta Lake uses a transaction log to provide ACID transactions, schema enforcement, versioning and consistent reads on top of data files.

Important: _delta_log is not a folder you should manually edit. Directly changing or deleting its files can corrupt the table's transaction history.

2. Perform INSERT, UPDATE and DELETE
Objective: Understand Delta Lake data modification operations.

INSERT
INSERT INTO customers VALUES
(5, 'Edward', 'Salem', 'edward@gmail.com', current_timestamp());


UPDATE

Change customer 2's city:
UPDATE customers
SET city = 'Chennai',
    updated_at = current_timestamp()
WHERE customer_id = 2;


DELETE

Delete customer 5:
DELETE FROM customers
WHERE customer_id = 5;


Verify:
SELECT * FROM customers ORDER BY customer_id;


Check the operations in the history:
DESCRIBE HISTORY customers;


What happens internally?

Delta Lake generally writes new data files for affected data and commits the corresponding changes to the transaction log. With deletion vectors enabled and supported for the operation, some row-level changes may be represented differently rather than immediately rewriting all affected files.

Interview answer: Delta Lake supports INSERT, UPDATE and DELETE using transactional commits. The transaction log ensures readers see a consistent table version.

3. Implement MERGE with duplicate source records
Objective: Perform an upsert and understand why duplicate source records matter.

First, create a source table:
CREATE OR REPLACE TABLE customer_updates (
    customer_id INT,
    customer_name STRING,
    city STRING,
    email STRING,
    updated_at TIMESTAMP
)
USING DELTA;


Insert source records, including duplicate customer IDs:
INSERT INTO customer_updates VALUES
(1, 'Arun Kumar', 'Bangalore', 'arun@gmail.com', current_timestamp()),
(2, 'Bala Kumar', 'Chennai', 'bala@gmail.com', current_timestamp()),
(2, 'Bala Kumar Updated', 'Trichy', 'bala@gmail.com', current_timestamp()),
(5, 'Edward', 'Salem', 'edward@gmail.com', current_timestamp());


Check duplicates:
SELECT customer_id, COUNT(*) AS record_count
FROM customer_updates
GROUP BY customer_id
HAVING COUNT(*) > 1;


Why should you remove duplicates?

If multiple source rows match the same target row, a MERGE can fail because the target row may be matched by multiple source rows. You should define a business rule to select the correct source record.

Deduplicate using the most recent update:
CREATE OR REPLACE TEMP VIEW customer_updates_dedup AS
SELECT
    customer_id,
    customer_name,
    city,
    email,
    updated_at
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY updated_at DESC, customer_name DESC
           ) AS rn
    FROM customer_updates
)
WHERE rn = 1;


Now perform MERGE:
MERGE INTO customers AS target
USING customer_updates_dedup AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET
        target.customer_name = source.customer_name,
        target.city = source.city,
        target.email = source.email,
        target.updated_at = source.updated_at

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        customer_name,
        city,
        email,
        updated_at
    )
    VALUES (
        source.customer_id,
        source.customer_name,
        source.city,
        source.email,
        source.updated_at
    );


Verify:
SELECT * FROM customers ORDER BY customer_id;


Expected behaviour:
- Existing customers are updated.
- New customer 5 is inserted.
- Only one source row per customer ID is used.
Interview answer: MERGE performs insert, update and optionally delete operations in one transactional operation. Before MERGE, I deduplicate the source based on a business key and a deterministic ordering column, such as the latest update timestamp.

Production warning: A timestamp tie needs a further deterministic tie-breaker, such as a source sequence number or ingestion ID. Don't use an arbitrary ordering rule if it can select the wrong business record.

4. Implement SCD Type 1
Objective: Update existing records without maintaining historical values.

SCD Type 1 overwrites old values with new values. It is useful when historical changes are not required.

Example: customer 1 moves from Chennai to Bangalore. We only want the latest city.

Create a source table:
CREATE OR REPLACE TABLE customer_type1_source (
    customer_id INT,
    customer_name STRING,
    city STRING,
    updated_at TIMESTAMP
)
USING DELTA;

INSERT INTO customer_type1_source VALUES
(1, 'Arun Kumar', 'Bangalore', current_timestamp()),
(2, 'Bala Kumar', 'Trichy', current_timestamp()),
(6, 'Franklin', 'Theni', current_timestamp());


Deduplicate source records first if necessary:
CREATE OR REPLACE TEMP VIEW customer_type1_dedup AS
SELECT customer_id, customer_name, city, updated_at
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY updated_at DESC, customer_name DESC
           ) AS rn
    FROM customer_type1_source
)
WHERE rn = 1;


Perform SCD Type 1:
MERGE INTO customers AS target
USING customer_type1_dedup AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET
        target.customer_name = source.customer_name,
        target.city = source.city,
        target.updated_at = source.updated_at

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        customer_name,
        city,
        updated_at
    )
    VALUES (
        source.customer_id,
        source.customer_name,
        source.city,
        source.updated_at
    );


Verify:
SELECT * FROM customers ORDER BY customer_id;


Expected result: Customer 1's old city is replaced with Bangalore. The old value is not maintained in the current table.

Interview answer: SCD Type 1 overwrites existing dimension values. It is used when only the latest state matters and historical tracking is unnecessary.

5. Implement SCD Type 2
Objective: Maintain complete historical records when a customer's attributes change.

SCD Type 2 does not overwrite the old record. It expires the old version and inserts a new version.

Typical columns:
Column	Purpose
customer_id	Business key
customer_name	Customer name
city	Customer city
start_date	When this version became active
end_date	When this version expired
is_current	Identifies the active version

Create the dimension table:
CREATE OR REPLACE TABLE dim_customer (
    customer_id INT,
    customer_name STRING,
    city STRING,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN
)
USING DELTA;


Insert initial records:
INSERT INTO dim_customer VALUES
(1, 'Arun', 'Chennai', DATE '2026-01-01', DATE '9999-12-31', TRUE),
(2, 'Bala', 'Madurai', DATE '2026-01-01', DATE '9999-12-31', TRUE);


Create a source table:
CREATE OR REPLACE TABLE customer_scd2_source (
    customer_id INT,
    customer_name STRING,
    city STRING
)
USING DELTA;

INSERT INTO customer_scd2_source VALUES
(1, 'Arun', 'Bangalore'),
(2, 'Bala', 'Madurai'),
(3, 'Charan', 'Coimbatore');


Step 1: Expire changed records
MERGE INTO dim_customer AS target
USING (
    SELECT *
    FROM customer_scd2_source
) AS source

ON target.customer_id = source.customer_id
AND target.is_current = TRUE

WHEN MATCHED
AND (
    NOT (target.city <=> source.city)
    OR NOT (target.customer_name <=> source.customer_name)
)
THEN UPDATE SET
    target.end_date = current_date(),
    target.is_current = FALSE;


<=> is Spark SQL's null-safe equality operator. It handles comparisons involving NULL values.

Step 2: Insert new versions

Use a staged source containing changed records and brand-new customers:
CREATE OR REPLACE TEMP VIEW scd2_new_versions AS

SELECT
    source.customer_id,
    source.customer_name,
    source.city
FROM customer_scd2_source source

LEFT JOIN dim_customer target
    ON source.customer_id = target.customer_id
    AND target.is_current = TRUE

WHERE target.customer_id IS NULL
   OR NOT (target.city <=> source.city)
   OR NOT (target.customer_name <=> source.customer_name);


Insert the new versions:
INSERT INTO dim_customer
SELECT
    customer_id,
    customer_name,
    city,
    current_date(),
    DATE '9999-12-31',
    TRUE
FROM scd2_new_versions;


Verify the result:
SELECT *
FROM dim_customer
ORDER BY customer_id, start_date;


Expected logical result:
Customer	City	Current?
1	Chennai	false
1	Bangalore	true
2	Madurai	true
3	Coimbatore	true

Interview answer: SCD Type 2 preserves historical changes by expiring the previous active record and inserting a new active record. We commonly use start_date, end_date and is_current columns.

Production warning: These two SQL operations are separate transactions. A production SCD2 pipeline must handle retries, concurrent runs, late-arriving changes, multiple changes for one business key and failure between expiration and insertion. Use an idempotent processing design and appropriate orchestration or staging strategy. Don't claim that the two statements are one atomic MERGE.

6. Read historical versions using Time Travel
Objective: Retrieve older versions of a Delta table.

Check table history:
DESCRIBE HISTORY customers;


You will see information such as:
- Version number
- Timestamp
- Operation
- Operation parameters
- User metadata
Read a specific version:
SELECT *
FROM customers VERSION AS OF 2;


Read the table as of a timestamp:
SELECT *
FROM customers TIMESTAMP AS OF '2026-11-16 08:00:00';


Replace the sample timestamp with an actual timestamp from your table history.

Compare two versions:
SELECT *
FROM customers VERSION AS OF 1

EXCEPT ALL

SELECT *
FROM customers VERSION AS OF 2;


Restore the table to an older version:
RESTORE TABLE customers TO VERSION AS OF 2;


Check history again:
DESCRIBE HISTORY customers;


Important difference:
- VERSION AS OF reads historical data without changing the current table.
- RESTORE creates a new current version whose contents reflect the selected older version.
Interview answer: Delta Lake Time Travel allows us to query previous table versions using version numbers or timestamps. It is useful for debugging, auditing and recovering from accidental modifications.

Retention warning: Time Travel depends on the required data files and transaction-log history still being available. VACUUM can permanently remove old data files.

7. Enable and read Change Data Feed (CDF)
Objective: Capture row-level changes between Delta table versions.

Change Data Feed helps downstream pipelines process only inserted, updated and deleted records instead of reading the entire table again.

Enable CDF:
ALTER TABLE customers
SET TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);


Make some changes after enabling CDF:
INSERT INTO customers VALUES
(10, 'Kumar', 'Theni', 'kumar@gmail.com', current_timestamp());

UPDATE customers
SET city = 'Chennai'
WHERE customer_id = 10;

DELETE FROM customers
WHERE customer_id = 10;


Check the table history:
DESCRIBE HISTORY customers;


Read changes between versions:
SELECT *
FROM table_changes('workspace.delta_practice.customers', 5, 8);


Use actual start and end versions from DESCRIBE HISTORY. The example versions are illustrative.

CDF returns metadata columns including:
Metadata column	Meaning
_change_type	Type of change
_commit_version	Version in which the change was committed
_commit_timestamp	Commit timestamp

Common change types:
- insert
- delete
- update_preimage
- update_postimage
Example:
SELECT
    customer_id,
    customer_name,
    city,
    _change_type,
    _commit_version
FROM table_changes(
    'workspace.delta_practice.customers',
    5,
    8
);


Interview answer: CDF captures row-level changes in a Delta table. It is useful for incremental ETL, CDC pipelines, auditing and synchronizing downstream tables.

Important: CDF must be enabled for the relevant changes to be captured. It does not automatically mean that every old historical change is available forever. Retention and table configuration matter.

8. Run OPTIMIZE and ZORDER
Objective: Improve query performance by reducing small-file overhead and organizing data for efficient data skipping.

Imagine an orders table containing millions of records.

Create a sample table:
CREATE OR REPLACE TABLE orders (
    order_id BIGINT,
    customer_id INT,
    order_date DATE,
    city STRING,
    amount DECIMAL(10,2)
)
USING DELTA;


In a real project, load your order dataset into this table before optimizing.

Check table details:
DESCRIBE DETAIL orders;


Run OPTIMIZE:
OPTIMIZE orders;


OPTIMIZE compacts small files into fewer, larger files. The exact output depends on the data size, file sizes and runtime settings.

Run ZORDER:
OPTIMIZE orders
ZORDER BY (customer_id);


For queries frequently filtering on customer ID:
SELECT *
FROM orders
WHERE customer_id = 1001;


ZORDER co-locates related values to improve data skipping.

Interview answer: OPTIMIZE reduces small-file problems by compacting files. ZORDER organizes data to improve data skipping for commonly filtered columns.

Don't confuse these:
- OPTIMIZE: file compaction and layout optimization.
- ZORDER: data organization within the optimized files.
- Partitioning: physical separation of data into partitions.
- Liquid Clustering: a newer clustering approach that manages data layout without requiring traditional partitioning or ZORDER.

9. Convert a ZORDER table to Liquid Clustering
Objective: Understand the modern Databricks data-layout approach.

Suppose the table was previously optimized using:
OPTIMIZE orders
ZORDER BY (customer_id);


For a supported Delta table and Databricks runtime, enable Liquid Clustering using:
ALTER TABLE orders
CLUSTER BY (customer_id, order_date);


Then run:
OPTIMIZE orders;


Check the table definition:
DESCRIBE DETAIL orders;


Check clustering information:
SHOW CREATE TABLE orders;


Important: This changes the clustering definition; it does not instantly reorganize all existing files. Run OPTIMIZE to apply the new layout. Availability and conversion behaviour depend on your Databricks runtime and table features.

If you want to stop using clustering:
ALTER TABLE orders
CLUSTER BY NONE;


Use only if supported by your environment and after checking the implications.

ZORDER vs Liquid Clustering
ZORDER	Liquid Clustering
Uses Z-order data organization	Uses clustering columns
Applied through OPTIMIZE ... ZORDER BY	Defined using CLUSTER BY
Requires choosing columns for ZORDER	Supports evolving clustering choices
Traditional optimization approach	Modern layout approach
Not designed to be combined with Liquid Clustering on the same table	Use OPTIMIZE to maintain clustering

Interview answer: Liquid Clustering is a modern data-layout optimization technique in Databricks. It helps organize data for data skipping and supports changing clustering columns more flexibly than traditional partitioning and ZORDER.

10. Compare CTAS, DEEP CLONE and SHALLOW CLONE
Objective: Understand three ways to create tables from existing data.

Assume your source table is:
workspace.delta_practice.orders


A. CTAS – Create Table As Select
CREATE OR REPLACE TABLE orders_ctas
USING DELTA
AS
SELECT *
FROM orders;


Verify:
SELECT COUNT(*) FROM orders_ctas;


Explanation: CTAS creates a new table using the result of a query. The new table has its own data files. It does not maintain a live link to the source table.

Use cases:
- Creating reporting tables.
- Creating transformed datasets.
- Creating filtered or aggregated tables.
Example:
CREATE OR REPLACE TABLE high_value_orders
USING DELTA
AS
SELECT *
FROM orders
WHERE amount > 10000;


B. DEEP CLONE
CREATE OR REPLACE TABLE orders_deep_clone
DEEP CLONE orders;


Explanation: Deep Clone copies the source table's data files and metadata into an independent target table.

Verify:
SELECT COUNT(*) FROM orders_deep_clone;


If you modify the source, the deep clone does not automatically receive those changes.

Use cases:
- Independent testing environments.
- Creating a backup copy.
- Migrating tables.
C. SHALLOW CLONE
CREATE OR REPLACE TABLE orders_shallow_clone
SHALLOW CLONE orders;


Explanation: Shallow Clone creates a separate table with its own Delta metadata, but initially references the source table's existing data files instead of copying all of them.

Use cases:
- Fast development copies.
- Testing changes without immediately duplicating all source data.
- Space-efficient copies.
Critical limitation: A shallow clone depends on the source data files remaining available. If source files are removed by VACUUM, the shallow clone may become unreadable for data it still references.
Comparison
Feature	CTAS	DEEP CLONE	SHALLOW CLONE
Copies data into new files	Yes	Yes	No, initially
Independent table metadata	Yes	Yes	Yes
Copies source table history	No	Clone history is not a full copy of source history	No
Depends on source data files	No	No	Yes
Typical use	Transformation	Independent copy	Fast test copy

Interview answer: CTAS creates a new table from a query result. Deep Clone copies the data and metadata into an independent table. Shallow Clone copies metadata while referencing the original data files, making it faster and more storage-efficient initially.

11. Run VACUUM DRY RUN and understand retention
Objective: Understand how Delta Lake removes obsolete data files.

When you update or delete records, Delta Lake may create new data files and leave older files physically present. Those older files can be required for Time Travel or concurrent readers.

VACUUM removes unreferenced data files after the retention period.

Check the table history:
DESCRIBE HISTORY orders;


Run VACUUM in dry-run mode:
VACUUM orders DRY RUN;


What does DRY RUN do?

It lists files that would be eligible for deletion. It does not delete them.

Actual VACUUM:
VACUUM orders;


The default deleted-file retention threshold is generally 7 days (168 hours).

You can explicitly specify a retention period:
VACUUM orders RETAIN 168 HOURS;


Important: Do not reduce retention casually. Delta Lake has a safety check that protects against unsafe retention periods. Disabling that check can expose concurrent readers and writers to failures.

Check the retention configuration:
SHOW TBLPROPERTIES orders;


Relevant properties include:
delta.deletedFileRetentionDuration
delta.logRetentionDuration


These are different:
Setting	Default commonly used	Purpose
Deleted-file retention	7 days	How long obsolete data files are retained before VACUUM can remove them
Log retention	30 days	How long transaction-log history is retained

Interview answer: VACUUM physically deletes obsolete data files that are no longer needed by the current table snapshot, subject to retention rules. It helps reclaim storage, but can reduce Time Travel availability.

Production warning: Before VACUUM, check active streams, long-running jobs, concurrent readers, clone dependencies and recovery requirements. A successful DRY RUN is not a guarantee that deleting those files is safe for every downstream consumer.

12. Explain the complete E-Commerce Delta architecture
Objective: Explain how all the Delta Lake concepts work together in a real-world Data Engineering project.

Your project uses a Medallion Architecture with Bronze, Silver and Gold layers.








figlab.ai


A. Source data
An e-commerce company receives data from different business systems:
Source	Example data
Customers	Customer ID, name, email
Products	Product ID, product name, price
Orders	Order ID, customer ID, order date
Order Items	Product ID, quantity, unit price
Payments	Payment method, payment status
Shipments	Shipping status, delivery date
Inventory	Product stock and warehouse quantity

These source files are stored as CSV files in your landing volume.

Example project location:
CATALOG = "workspace"SCHEMA = "default"VOLUME = "data_volume"BASE_PATH = (    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ecommerce_lakehouse")LANDING_PATH = f"{BASE_PATH}/landing"CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints"SCHEMA_PATH = f"{BASE_PATH}/schemas"



B. Bronze layer – Raw data
Raw ingestion
Purpose:
- Ingest source files.
- Preserve raw data.
- Add ingestion metadata.
- Track processing using Auto Loader.
- Store data in Delta format.
Example:
from pyspark.sql.functions import current_timestamp, input_file_namesource_path = f"{LANDING_PATH}/customers"df = (    spark.read    .option("header", True)    .csv(source_path))bronze_df = (    df.withColumn("ingestion_timestamp", current_timestamp())      .withColumn("source_file", input_file_name()))(    bronze_df.write    .format("delta")    .mode("append")    .saveAsTable("workspace.ecommerce_bronze.customers"))



For incremental file ingestion, Auto Loader can be used:
source_path = f"{LANDING_PATH}/customers"checkpoint_path = f"{CHECKPOINT_PATH}/customers"schema_path = f"{SCHEMA_PATH}/customers"df = (    spark.readStream    .format("cloudFiles")    .option("cloudFiles.format", "csv")    .option("header", "true")    .option("cloudFiles.schemaLocation", schema_path)    .load(source_path))bronze_df = (    df.withColumn("ingestion_timestamp", current_timestamp())      .withColumn("source_file", input_file_name()))(    bronze_df.writeStream    .format("delta")    .option("checkpointLocation", checkpoint_path)    .trigger(availableNow=True)    .toTable("workspace.ecommerce_bronze.customers"))



C. Silver layer – Cleaned and validated data
Cleansing and transformation
Purpose:
- Remove duplicates.
- Handle missing values.
- Validate data types.
- Apply business rules.
- Check referential integrity.
- Quarantine invalid records.
- Perform incremental upserts using MERGE.
Example:
CREATE OR REPLACE TABLE workspace.ecommerce_silver.customers
USING DELTA
AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY ingestion_timestamp DESC
           ) AS rn
    FROM workspace.ecommerce_bronze.customers
    WHERE customer_id IS NOT NULL
)
WHERE rn = 1;


Incremental updates can be applied using MERGE:
MERGE INTO workspace.ecommerce_silver.customers AS target
USING customer_updates_dedup AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;


This example assumes compatible source and target schemas. In production, explicitly list the columns and validate the source data before merging.
D. Gold layer – Business-ready data
Analytics
Purpose:
- Calculate business KPIs.
- Create reporting datasets.
- Build dashboards.
- Support business decisions.
Example: Daily sales summary.
CREATE OR REPLACE TABLE workspace.ecommerce_gold.daily_sales
USING DELTA
AS
SELECT
    order_date,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(amount) AS total_revenue
FROM workspace.ecommerce_silver.orders
GROUP BY order_date;


Example dashboard metrics:
Dashboard	Business metrics
Executive KPI	Total revenue, orders, average order value
Sales	Daily revenue, monthly growth
Product	Best-selling products, product revenue
Customer	New customers, repeat customers, lifetime value
Payment	Successful and failed payments
Shipment	Delivered, delayed and pending shipments
Inventory	Available stock, low-stock products

E. Performance optimization
Apply Delta Lake features where they solve actual problems:
Feature	Where it helps
MERGE	Incremental customer and order updates
Time Travel	Debugging incorrect transformations
CDF	Incremental downstream processing
OPTIMIZE	Small-file compaction
ZORDER	Data skipping on frequently filtered columns
Liquid Clustering	Adaptive data layout for supported tables
VACUUM	Storage cleanup
Deep Clone	Independent testing or backup copy
SCD Type 2	Customer history tracking

F. Orchestration
A typical workflow:
Source CSV files

Customers · Products · Orders · Payments · Shipments · Inventory


Auto Loader / Databricks Jobs

Incremental ingestion · Schema tracking · Checkpoints


Bronze – Raw Delta tables

Raw records · Ingestion metadata


Silver – Clean Delta tables

Deduplication · Validation · MERGE · SCD · Quarantine


Gold – Analytics Delta tables

Daily sales · Product performance · Customer lifetime value · KPIs


Databricks SQL Dashboards

Executive reporting · Business analytics


Interview answer – End-to-end project explanation:

"I developed an E-Commerce Lakehouse using Databricks, PySpark, Delta Lake and SQL. Data from customers, products, orders, payments, shipments and inventory was ingested from CSV files using Auto Loader into the Bronze layer. In the Silver layer, I performed data cleansing, deduplication, validation, referential integrity checks and incremental upserts using MERGE. I used SCD Type 2 where historical customer changes needed to be maintained. In the Gold layer, I created business-ready aggregated tables for sales, product performance, customer lifetime value, payment metrics and inventory. I used Delta Lake features such as ACID transactions, Time Travel, CDF and OPTIMIZE to support reliable incremental processing, debugging and query performance. Finally, I connected the Gold tables to Databricks SQL dashboards for business reporting."

Be precise in an interview: Only claim that you implemented a feature if you actually executed it in your project. If you only practiced it separately, say that it was part of your hands-on learning rather than production implementation.